In [5]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from datasets import Dataset
import os
from dotenv import load_dotenv
from BaselineModel import BaselineModel


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
load_dotenv()
DEFAULT_DETECTION_CLASS = 'no'

In [13]:
detect_train_df = pd.read_csv('../data/detect_train.csv')
detect_train_dataset = Dataset.from_pandas(detect_train_df)
detect_train_dataset.info.metadata = {'name': 'deduplicated_detect_train'}

detect_test_df = pd.read_csv('../data/detect_test.csv')
detect_test_dataset = Dataset.from_pandas(detect_test_df)
detect_test_dataset.info.metadata = {'name': 'deduplicated_detect_test'}


In [8]:
import shutil
BASE_MAT_DIRECTORY = os.getenv('BASE_MAT_DIRECTORY')
MAT_NEW_INPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/trained/input'
MAT_NEW_OUTPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/trained/output'
shutil.copytree('../config/baseline/dic', MAT_NEW_INPUT_DIRECTORY + '/dic', dirs_exist_ok=True)
os.makedirs(os.path.join(MAT_NEW_INPUT_DIRECTORY, 'origin'), exist_ok=True)
os.makedirs(MAT_NEW_OUTPUT_DIRECTORY, exist_ok=True)
for kv in [{'train': detect_train_df}, {'test': detect_test_df}, {'merged': pd.concat([detect_train_df.assign(project='train'), detect_test_df.assign(project='test')])}]:
    for df_name, df in kv.items():
        if df_name == 'merged':
            df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/comments', index=False, header=False)
            df['label'].str.lower().map({'yes': 'SATD', 'no': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/labels', index=False, header=False)
            df['project'].to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/projects', index=False, header=False)
        else:
            df['text'].str.replace('\n', '\t').to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/data--{df_name}.txt', index=False, header=False)
            df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_NEW_INPUT_DIRECTORY}/origin/label--{df_name}.txt', index=False, header=False)




MAT_PRETRAINED_INPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/pretrained/input'
MAT_PRETRAINED_OUTPUT_DIRECTORY = f'{BASE_MAT_DIRECTORY}/pretrained/output'
os.makedirs(MAT_PRETRAINED_OUTPUT_DIRECTORY, exist_ok=True)
shutil.copytree('../config/baseline', MAT_PRETRAINED_INPUT_DIRECTORY, dirs_exist_ok=True)

detect_test_df['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/data--test.txt', index=False, header=False)
detect_test_df['label'].str.lower().map({'yes': 'positive', 'no': 'negative'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', index=False, header=False)
df1 = pd.DataFrame(open('../config/baseline/origin/data--train.txt').read().splitlines(), columns=["text"])
df2 = detect_test_df[['text']]
pd.concat([df1, df2])['text'].str.replace('\n', '\t').to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/comments', index=False, header=False)

df1 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--train.txt', header=None, names=['label']).assign(project='train')
df2 = pd.read_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/label--test.txt', header=None, names=['label']).assign(project='test')
df = pd.concat([df1, df2])
df['label'].map({'positive': 'SATD', 'negative': 'WITHOUT_CLASSIFICATION'}).to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/labels', columns=['label'], index=False, header=False)

df.to_csv(f'{MAT_PRETRAINED_INPUT_DIRECTORY}/origin/projects', columns=['project'], index=False, header=False)



# Potdar Pattern

In [18]:
pattern_model = BaselineModel('detect', f'pretrained-potdar-Pattern', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pattern_model.fit(detect_train_dataset)
pattern_model.predict(detect_test_dataset)

detect with pretrained-potdar-Pattern
True
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
233, 5573, 78, 112431, 0.749, 0.040, 0.076, 0.934, 14.267
3, 109, 2, 6852, 0.600, 0.027, 0.051, 0.973, 36.318

Test Result:
              precision    recall  f1-score   support

          no      0.984     1.000     0.992      6855
         yes      0.600     0.027     0.051       112

    accuracy                          0.984      6967
   macro avg      0.792     0.513     0.522      6967
weighted avg      0.978     0.984     0.977      6967



'../cache/output/tmp/September 11, 2025, 14:14:40$detect_pretrained-potdar-Pattern.csv'

In [19]:
trained_pattern_model = BaselineModel('detect', f'trained-potdar-Pattern', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
trained_pattern_model.fit(detect_train_dataset)
trained_pattern_model.predict(detect_test_dataset)

detect with trained-potdar-Pattern
True
Running model Pattern in MTO
Preparing data for Pattern
Pattern prediction finished!
Method: Pattern
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
14, 390, 6, 25177, 0.700, 0.035, 0.066, 0.977, 43.334
3, 109, 2, 6852, 0.600, 0.027, 0.051, 0.973, 36.318

Test Result:
              precision    recall  f1-score   support

          no      0.984     1.000     0.992      6855
         yes      0.600     0.027     0.051       112

    accuracy                          0.984      6967
   macro avg      0.792     0.513     0.522      6967
weighted avg      0.978     0.984     0.977      6967



'../cache/output/tmp/September 11, 2025, 14:14:44$detect_trained-potdar-Pattern.csv'

# Text Mining

In [24]:
pretrained_tm_model = BaselineModel('detect', f'pretrained-TM', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pretrained_tm_model.fit(detect_train_dataset)
pretrained_tm_model.predict(detect_test_dataset)

detect with pretrained-TM
True
Running model TM in MTO
Preparing data for Pattern
../cache/baseline/pretrained/input/tm/data--train.arff
../cache/baseline/pretrained/input/tm/data--test.arff
Target: train, ../cache/baseline/pretrained/input/tm/data--train.arff
Target: test, ../cache/baseline/pretrained/input/tm/data--test.arff
Method: TM
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
4236, 1570, 3866, 108643, 0.523, 0.730, 0.609, 0.906, 9.654
45, 67, 328, 6526, 0.121, 0.402, 0.186, 0.867, 6.504

Test Result:
              precision    recall  f1-score   support

          no      0.990     0.952     0.971      6855
         yes      0.121     0.402     0.186       112

    accuracy                          0.943      6967
   macro avg      0.555     0.677     0.578      6967
weighted avg      0.976     0.943     0.958      6967



'../cache/output/tmp/September 11, 2025, 14:34:30$detect_pretrained-TM.csv'

In [25]:
trained_tm_model = BaselineModel('detect', f'trained-TM', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
trained_tm_model.fit(detect_train_dataset)
trained_tm_model.predict(detect_test_dataset)

detect with trained-TM
True
Running model TM in MTO
Preparing data for Pattern
../cache/baseline/trained/input/tm/data--train.arff
../cache/baseline/trained/input/tm/data--test.arff
Target: train, ../cache/baseline/trained/input/tm/data--train.arff
Target: test, ../cache/baseline/trained/input/tm/data--test.arff
Method: TM
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
238, 166, 467, 24716, 0.338, 0.589, 0.429, 0.953, 20.381
65, 47, 201, 6653, 0.244, 0.580, 0.344, 0.934, 14.198

Test Result:
              precision    recall  f1-score   support

          no      0.993     0.971     0.982      6855
         yes      0.244     0.580     0.344       112

    accuracy                          0.964      6967
   macro avg      0.619     0.776     0.663      6967
weighted avg      0.981     0.964     0.971      6967



'../cache/output/tmp/September 11, 2025, 14:35:36$detect_trained-TM.csv'

# NLP

In [ ]:
pretrained_nlp_model = BaselineModel('detect', f'pretrained-NLP', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pretrained_nlp_model.fit(detect_train_dataset)
pretrained_nlp_model.predict(detect_test_dataset)

In [ ]:
trained_nlp_model = BaselineModel('detect', f'trained-NLP', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
trained_nlp_model.fit(detect_train_dataset)
trained_nlp_model.predict(detect_test_dataset)

# MAT

In [22]:
pretrained_mat_model = BaselineModel('detect', f'pretrained-MAT', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
pretrained_mat_model.fit(detect_train_dataset)
pretrained_mat_model.predict(detect_test_dataset)

detect with pretrained-MAT
True
Running model MAT in MTO
Preparing data for Pattern
../cache/baseline/pretrained/input/tm/data--train.arff
../cache/baseline/pretrained/input/tm/data--test.arff
MAT prediction finished!
Method: MAT
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
4452, 1354, 1030, 111479, 0.812, 0.767, 0.789, 0.940, 15.549
59, 53, 2, 6852, 0.967, 0.527, 0.682, 0.983, 59.157

Test Result:
              precision    recall  f1-score   support

          no      0.992     1.000     0.996      6855
         yes      0.967     0.527     0.682       112

    accuracy                          0.992      6967
   macro avg      0.980     0.763     0.839      6967
weighted avg      0.992     0.992     0.991      6967



'../cache/output/tmp/September 11, 2025, 14:16:13$detect_pretrained-MAT.csv'

In [23]:
trained_mat_model = BaselineModel('detect', f'trained-MAT', {'yes', 'no'}, DEFAULT_DETECTION_CLASS)
trained_mat_model.fit(detect_train_dataset)
trained_mat_model.predict(detect_test_dataset)

detect with trained-MAT
True
Running model MAT in MTO
Preparing data for Pattern
../cache/baseline/trained/input/tm/data--train.arff
../cache/baseline/trained/input/tm/data--test.arff
MAT prediction finished!
Method: MAT
TP, FN, FP, TN, P    , R    , F1   , ER   , RI
255, 149, 14, 25169, 0.948, 0.631, 0.758, 0.983, 59.038
59, 53, 2, 6852, 0.967, 0.527, 0.682, 0.983, 59.157

Test Result:
              precision    recall  f1-score   support

          no      0.992     1.000     0.996      6855
         yes      0.967     0.527     0.682       112

    accuracy                          0.992      6967
   macro avg      0.980     0.763     0.839      6967
weighted avg      0.992     0.992     0.991      6967



'../cache/output/tmp/September 11, 2025, 14:16:25$detect_trained-MAT.csv'